In [24]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import urllib.request  # Built-in module to download the text labels
import timm  # The gold-standard library for Vision Transformers in PyTorch
import torchvision.transforms as transforms

'''
from google.colab import drive
drive.mount('/content/drive')
'''

#1. preprocessing
preprocess_pipeline = transforms.Compose([
    transforms.Resize((256, 128)),
    transforms.ToTensor(),
    # Standard normalization constants used by ImageNet pre-trained models
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def process_image(image_path):
    image = Image.open(image_path).convert('RGB')
    tensor = preprocess_pipeline(image)
    tensor = tensor.unsqueeze(0)
    return tensor

# 3. LOAD YOUR REAL IMAGES
img1_tensor = process_image("./media/cat1.webp")
img2_tensor = process_image("./media/cat2.webp")

In [25]:
# 2. INITIALIZE THE COMPRESSED FEATURE EXTRACTOR
resnet50 = models.resnet50(pretrained=True)
resnet50.fc = torch.nn.Identity() # Remove classification layer to get raw features
resnet50.eval()

# 4. EXTRACT THE MATHEMATICAL FINGERPRINTS (EMBEDDINGS)
with torch.no_grad():
    features_1 = resnet50(img1_tensor) # Outputs a vector of shape [1, 2048]
    features_2 = resnet50(img2_tensor) # Outputs a vector of shape [1, 2048]

# 5. CALCULATE SIMILARITY SCORE
# We use Cosine Similarity to measure the angle between the two 2048-dimensional vectors
similarity_fn = torch.nn.CosineSimilarity(dim=1)
score = similarity_fn(features_1, features_2)

print(f"Similarity Score between the two images: {score.item():.4f}")
# A score close to 1.0 means the model recognizes highly identical visual features!

Similarity Score between the two images: 0.6913


In [26]:
# --- 0. DOWNLOAD IMAGENET TEXT LABELS ---
# We fetch the official list of 1,000 object categories that ResNet50 knows
labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
urllib.request.urlretrieve(labels_url, "./media/imagenet_classes.txt")

with open("./media/imagenet_classes.txt", "r") as f:
    categories = [line.strip() for line in f.readlines()]

# --- 2. INITIALIZE THE STOCK CLASS MODEL ---
resnet50 = models.resnet50(pretrained=True)
# REMOVED: resnet50.fc = torch.nn.Identity() <- We KEEP the classifier layer active!
resnet50.eval()

# --- 4. EXTRACT CONFIDENCE SCORES ---
with torch.no_grad():
    # These output 1,000 raw confidence scores per image instead of a 2,048 vector!
    output_1 = resnet50(img1_tensor)
    output_2 = resnet50(img2_tensor)

# --- 5. INTERPRET THE RESULTS (THE MEMBERSHIP VERDICT) ---
# We use torch.max() to pull the highest confidence value and its array index location
prob_1, index_1 = torch.max(output_1, dim=1)
prob_2, index_2 = torch.max(output_2, dim=1)

print("--- CLASSIFICATION RESULTS ---")
print(f"Image 1 Predicted Category: {categories[index_1.item()]} (Raw Score: {prob_1.item():.4f})")
print(f"Image 2 Predicted Category: {categories[index_2.item()]} (Raw Score: {prob_2.item():.4f})")

--- CLASSIFICATION RESULTS ---
Image 1 Predicted Category: tiger cat (Raw Score: 20.5276)
Image 2 Predicted Category: Egyptian cat (Raw Score: 13.3859)


In [27]:
# 2. INITIALIZE THE VISION TRANSFORMER (The Black Box Switch)
# We load 'vit_base_patch16_224' pre-trained on ImageNet.
# setting 'num_classes=0' automatically drops the final classification layer!
# This leaves us with a pure feature extractor out-of-the-box.
vit_model = timm.create_model(
    'vit_base_patch16_224', 
    pretrained=True, 
    num_classes=0, 
    img_size=(256, 128) 
)
vit_model.eval()

# 4. EXTRACT THE MATHEMATICAL FINGERPRINTS
with torch.no_grad():
    features_1 = vit_model(img1_tensor)  # Passes patch tokens through self-attention layers
    features_2 = vit_model(img2_tensor)

# 5. INSPECT AND MEASURE PATTERN SIMILARITY
print("ViT Feature Vector Shape:", features_1.shape)

similarity_fn = torch.nn.CosineSimilarity(dim=1)
score = similarity_fn(features_1, features_2)

print(f"ViT Cosine Similarity Score: {score.item():.4f}")


ViT Feature Vector Shape: torch.Size([1, 768])
ViT Cosine Similarity Score: 0.4588
